

---

# **TRABAJO PARCIAL - TEORÍA DE COMPILADORES**
Sección : 13962



---


**INTEGRANTES**

Vanessa Barrientos Villalta - U202413287

Matias Javier Del Castillo Mendoza - U202411305

Vivianne Fatima Ríos Hasegawa - U202410135



---



**Instalación de ANTLR**

In [ ]:
!pip install antlr4-python3-runtime==4.13.2 -q
!curl -s -O https://www.antlr.org/download/antlr-4.13.2-complete.jar

**ANALIZADOR LÉXICO**

Creación del archivo "SoundSpecLexer.g4"

In [ ]:
grammar = r'''
lexer grammar SoundSpecLexer;

// --- KEYWORDS ---
TRACK     : 'track';
EFFECT    : 'effect';
MIX       : 'mix';
BPM       : 'bpm';
WAVE      : 'wave';
DELAY     : 'delay';
DECAY     : 'decay';
NOTES     : 'notes';

// --- VALORES TIPADOS Y ENUMS ---
WAVE_TYPE    : 'SAWTOOTH' | 'SINE' | 'SQUARE' | 'TRIANGLE';
NOTE         : [A-G][#b]?[0-8];
FRACTION     : [0-9]+ '/' [0-9]+;
TIME_MS      : [0-9]+ 'ms';

// --- OPERADORES Y SÍMBOLOS ---
PIPE_OP      : '->';
ASSIGN       : '=';
COLON        : ':';
SEMI         : ';';
COMMA        : ',';
LBRACE       : '{';
RBRACE       : '}';
LBRACK       : '[';
RBRACK       : ']';

// --- LITERALES NUMÉRICOS E IDENTIFICADORES ---
FLOAT : [0-9]+ '.' [0-9]+;
INT   : [0-9]+;
ID    : [a-zA-Z_][a-zA-Z0-9_]*;

// --- REGLAS AUXILIARES ---
WS           : [ \t\r\n]+ -> skip;
COMMENT      : '//' ~[\r\n]* -> skip;
'''

with open("SoundSpecLexer.g4", "w") as f:
    f.write(grammar)

Compilar con ANTLR4

In [ ]:
!java -jar antlr-4.13.2-complete.jar -Dlanguage=Python3 SoundSpecLexer.g4

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


Pruebas del analizador léxico

In [ ]:
from antlr4 import InputStream
from antlr4.error.ErrorListener import ErrorListener
from SoundSpecLexer import SoundSpecLexer

class ListenerErrores(ErrorListener):
    def __init__(self, fase):
        super().__init__()
        self.fase = fase
        self.errores = []

    def syntaxError(self, recognizer, offendingSymbol, line, column, msg, e):
        self.errores.append({
            "fase": self.fase,
            "linea": line,
            "columna": column + 1,
            "mensaje": msg,
        })

def probar_analizador_lexico(cod_fuente, Nprueba):
    print(Nprueba)
    print("\nCódigo de entrada:" + cod_fuente)

    input_stream = InputStream(cod_fuente)
    lexer = SoundSpecLexer(input_stream)
    errores = ListenerErrores("LÉXICO")
    lexer.removeErrorListeners()
    lexer.addErrorListener(errores)
    tokens = lexer.getAllTokens()

    print(f"{'TOKEN':<18} | {'LEXEMA'}")
    print("-" * 40)
    for token in tokens:
        token_name = lexer.symbolicNames[token.type]
        print(f"{token_name:<18} | '{token.text}'")
    for error in errores.errores:
        print(f"[{error['fase']}] línea {error['linea']}, columna {error['columna']}: {error['mensaje']}")
    print("\n" + "=" * 50 + "\n")

# --- PRUEBA 1: Declaración de Track Básica ---
prueba1 = """
track BassLine {
    bpm: 120;
    wave: SAWTOOTH;
    notes: [ C2: 1/4, E2: 1/4, G2: 1/2 ];
}
"""

# --- PRUEBA 2: Declaración de Effect y Mix Simple ---
prueba2 = """
effect Reverb1 {
    delay: 200ms;
    decay: 0.5;
}

mix Song = BassLine -> Reverb1;
"""

# --- PRUEBA 3: Cadena de Efectos en Serie (Procesamiento Múltiple) ---
prueba3 = """
track SubBass {
    bpm: 120;
    wave: SINE;
    notes: [ C1: 1/2, G1: 1/2 ];
}

effect DelayCorto {
    delay: 150ms;
    decay: 0.4;
}

effect EcoLargo {
    delay: 400ms;
    decay: 0.2;
}

mix BajoProcesado = SubBass -> DelayCorto -> EcoLargo;
"""

# --- PRUEBA 4: Polifonía/Multi-Track y Diferentes Ondas ---
prueba4 = """
track LeadSintetizador {
    bpm: 140;
    wave: SQUARE;
    notes: [ C4: 1/8, E4: 1/8, G4: 1/4, B4: 1/2 ];
}

track PadAtmósfera {
    bpm: 140;
    wave: TRIANGLE;
    notes: [ C3: 1/1 ];
}

effect EspacioGrande {
    delay: 500ms;
    decay: 0.6;
}

mix MelodiaPrincipal = LeadSintetizador -> EspacioGrande;
mix FondoAmbiente = PadAtmósfera;
"""

# Ejecutar las pruebas
probar_analizador_lexico(prueba1, "PRUEBA 1: Definición de Pista Musical Básica")
probar_analizador_lexico(prueba2, "PRUEBA 2: Definición de Efecto y Mezcla Simple")
probar_analizador_lexico(prueba3, "PRUEBA 3: Cadena de Efectos Múltiples (Serie)")
probar_analizador_lexico(prueba4, "PRUEBA 4: Configuración Multi-Pista y Tipos de Onda")

PRUEBA 1: Definición de Pista Musical Básica

Código de entrada:
track BassLine {
    bpm: 120;
    wave: SAWTOOTH;
    notes: [ C2: 1/4, E2: 1/4, G2: 1/2 ];
}

TOKEN              | LEXEMA
----------------------------------------
TRACK              | 'track'
ID                 | 'BassLine'
LBRACE             | '{'
BPM                | 'bpm'
COLON              | ':'
INT                | '120'
SEMI               | ';'
WAVE               | 'wave'
COLON              | ':'
WAVE_TYPE          | 'SAWTOOTH'
SEMI               | ';'
NOTES              | 'notes'
COLON              | ':'
LBRACK             | '['
NOTE               | 'C2'
COLON              | ':'
FRACTION           | '1/4'
COMMA              | ','
NOTE               | 'E2'
COLON              | ':'
FRACTION           | '1/4'
COMMA              | ','
NOTE               | 'G2'
COLON              | ':'
FRACTION           | '1/2'
RBRACK             | ']'
SEMI               | ';'
RBRACE             | '}'


PRUEBA 2: Definición de Efecto

**Observaciones del analizador léxico**

- `PadAtmósfera` produce un error léxico porque `ID` solo admite letras ASCII, dígitos y guion bajo (`[a-zA-Z_][a-zA-Z0-9_]*`). La `ó` no pertenece a ningún token.
- Los nombres con forma de nota (`A1`, `C4`, `Bb2`) se reconocen como `NOTE` y los tipos de onda (`SINE`, `SQUARE`, `TRIANGLE`, `SAWTOOTH`) como `WAVE_TYPE`, porque esas reglas se declaran antes que `ID`. Por eso funcionan como palabras reservadas y no pueden usarse como identificadores.

**ANALIZADOR SINTÁCTICO**

El analizador verifica la estructura de SoundSpec con los tokens del analizador léxico y las mismas reglas utilizadas por el analizador semántico. Si el programa es válido, devuelve su árbol sintáctico; si hay errores, indica la fase, la línea y la columna (numeradas desde 1).

| Elemento | Estructura admitida |
| --- | --- |
| Programa | Una o más declaraciones `track`, `effect` o `mix`, seguidas del fin de la entrada. |
| Pista | `track ID { ... }`, con una o más propiedades `bpm`, `wave` o `notes`. |
| Propiedades de pista | `bpm: INT;`, `wave: WAVE_TYPE;` y `notes: [ ... ];`. |
| Notas | Una o más notas separadas por comas; cada nota tiene una fracción de duración y puede llevar `:`: `C4:1/4` o `C4 1/4`. |
| Efecto | `effect ID { ... }`, con una o más propiedades `delay: TIME_MS;` o `decay: FLOAT;`. |
| Mezcla | `mix ID = ID (-> ID)*;`. |

**Orden de ejecución:** ejecutar las celdas en orden desde la instalación de ANTLR y la creación del lexer. La gramática `SoundSpecParser.g4` se genera una sola vez y su parser se comparte con el analizador semántico.

La existencia de los identificadores, el rango audible y la suma de duraciones corresponden al análisis semántico. Las propiedades pueden aparecer en cualquier orden. El programa debe contener al menos una declaración; los bloques de pistas y efectos deben contener al menos una propiedad.


Creación del archivo `SoundSpecParser.g4` (lo usan el analizador sintáctico y el semántico)

In [ ]:
parser_grammar = r"""
parser grammar SoundSpecParser;
options { tokenVocab=SoundSpecLexer; }

program    : statement+ EOF ;
statement  : trackDecl | effectDecl | mixRule ;

trackDecl  : TRACK ID LBRACE trackProp+ RBRACE ;
trackProp  : BPM COLON INT SEMI
           | WAVE COLON WAVE_TYPE SEMI
           | NOTES COLON noteList SEMI ;
noteList   : LBRACK noteItem (COMMA noteItem)* RBRACK ;
noteItem   : NOTE COLON? FRACTION ;

effectDecl : EFFECT ID LBRACE effectProp+ RBRACE ;
effectProp : DELAY COLON TIME_MS SEMI
           | DECAY COLON FLOAT SEMI ;

mixRule    : MIX ID ASSIGN ID (PIPE_OP ID)* SEMI ;
"""

with open("SoundSpecParser.g4", "w", encoding="utf-8") as f:
    f.write(parser_grammar)
print("Gramática compartida SoundSpecParser.g4 creada.")


Gramática compartida SoundSpecParser.g4 creada.


Compilar con ANTLR4 (también genera `SoundSpecParserListener`, que usa el analizador semántico)

In [ ]:
!java -jar antlr-4.13.2-complete.jar -Dlanguage=Python3 -listener SoundSpecParser.g4

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


Implementación del analizador y reporte de errores

In [ ]:
from antlr4 import InputStream, CommonTokenStream
from SoundSpecLexer import SoundSpecLexer
from SoundSpecParser import SoundSpecParser

def analizar_sintaxis(cod_fuente):
    """Valida toda la entrada. No realiza comprobaciones semánticas."""
    lexer = SoundSpecLexer(InputStream(cod_fuente))
    errores_lexicos = ListenerErrores("LÉXICO")
    lexer.removeErrorListeners()
    lexer.addErrorListener(errores_lexicos)
    tokens = CommonTokenStream(lexer)
    tokens.fill()

    # Un carácter rechazado por el lexer no debe desaparecer silenciosamente.
    if errores_lexicos.errores:
        return {"valido": False, "errores": errores_lexicos.errores,
                "arbol": None, "arbol_texto": None}

    tokens.seek(0)
    parser = SoundSpecParser(tokens)
    errores_sintacticos = ListenerErrores("SINTÁCTICO")
    parser.removeErrorListeners()
    parser.addErrorListener(errores_sintacticos)
    arbol = parser.program()
    valido = parser.getNumberOfSyntaxErrors() == 0
    return {
        "valido": valido,
        "errores": errores_sintacticos.errores,
        "arbol": arbol if valido else None,
        "arbol_texto": arbol.toStringTree(recog=parser) if valido else None,
    }

def probar_analizador_sintactico(cod_fuente, titulo, mostrar_arbol=True):
    print(titulo)
    print("Código de entrada:\n" + cod_fuente)
    resultado = analizar_sintaxis(cod_fuente)
    if resultado["valido"]:
        print("Sin errores sintácticos.")
        if mostrar_arbol:
            print("Árbol sintáctico:", resultado["arbol_texto"])
    else:
        for error in resultado["errores"]:
            print(f"[{error['fase']}] línea {error['linea']}, "
                  f"columna {error['columna']}: {error['mensaje']}")
        print("La entrada no se acepta; no se devuelve un árbol válido.")
    print("\n" + "." * 50 + "\n")
    return resultado

Pruebas del analizador sintáctico: entradas válidas

In [ ]:
casos_sintacticos_validos = [
    ("Pista con notas y dos puntos", """
track BassLine {
    bpm: 120;
    wave: SAWTOOTH;
    notes: [C2:1/4, E2:1/4, G2:1/2];
}
"""),
    ("Notas sin dos puntos, admitidas por la gramática", """
track Melodia {
    bpm: 140;
    wave: SINE;
    notes: [C4 1/2, G4 1/2];
}
"""),
    ("Programa completo con cadena de efectos", """
track SubBass { bpm: 120; wave: SINE; notes: [C1:1/2, G1:1/2]; }
effect DelayCorto { delay: 150ms; decay: 0.3; }
effect EcoLargo { delay: 450ms; decay: 0.7; }
mix BajoProcesado = SubBass -> DelayCorto -> EcoLargo;
"""),
    ("Comentarios, sostenidos y bemoles", """
// El lexer reconoce alteraciones y omite comentarios.
track Alteraciones { notes: [C#4:1/2, Db4:1/2]; wave: TRIANGLE; bpm: 100; }
mix Salida = Alteraciones;
"""),
    ("Propiedades en otro orden", "track Orden { wave: SQUARE; bpm: 90; }"),
]

for titulo, codigo in casos_sintacticos_validos:
    resultado = probar_analizador_sintactico(codigo, titulo)
    assert resultado["valido"] and not resultado["errores"], titulo

print(f"Pruebas válidas superadas: {len(casos_sintacticos_validos)}.")

Pista con notas y dos puntos
Código de entrada:

track BassLine {
    bpm: 120;
    wave: SAWTOOTH;
    notes: [C2:1/4, E2:1/4, G2:1/2];
}

Sin errores sintácticos.
Árbol sintáctico: (program (statement (trackDecl track BassLine { (trackProp bpm : 120 ;) (trackProp wave : SAWTOOTH ;) (trackProp notes : (noteList [ (noteItem C2 : 1/4) , (noteItem E2 : 1/4) , (noteItem G2 : 1/2) ]) ;) })) <EOF>)

..................................................

Notas sin dos puntos, admitidas por la gramática
Código de entrada:

track Melodia {
    bpm: 140;
    wave: SINE;
    notes: [C4 1/2, G4 1/2];
}

Sin errores sintácticos.
Árbol sintáctico: (program (statement (trackDecl track Melodia { (trackProp bpm : 140 ;) (trackProp wave : SINE ;) (trackProp notes : (noteList [ (noteItem C4 1/2) , (noteItem G4 1/2) ]) ;) })) <EOF>)

..................................................

Programa completo con cadena de efectos
Código de entrada:

track SubBass { bpm: 120; wave: SINE; notes: [C1:1/2, G1:1/2]; }

Pruebas del analizador sintáctico: entradas inválidas

Se comprueban delimitadores, tipos de tokens, listas de notas, cadenas de efectos y consumo completo de la entrada. También se incluyen dos errores léxicos para asegurar que no se acepten caracteres desconocidos. Con el lexer existente, los identificadores solo admiten letras ASCII, dígitos y guion bajo; por eso `PadAtmósfera` produce un error léxico, no un error sintáctico.

In [ ]:
casos_sintacticos_invalidos = [
    ("Programa vacío", "", "SINTÁCTICO"),
    ("Pista sin propiedades", "track Melodia { }", "SINTÁCTICO"),
    ("Asignación incorrecta de una propiedad", "track Melodia { wave = SINE; }", "SINTÁCTICO"),
    ("Falta punto y coma", "track Melodia { bpm: 120 }", "SINTÁCTICO"),
    ("Falta llave de cierre", "track Melodia { bpm: 120;", "SINTÁCTICO"),
    ("Falta coma entre notas", "track Melodia { notes: [C4:1/2 G4:1/2]; }", "SINTÁCTICO"),
    ("Lista de notas vacía", "track Melodia { notes: []; }", "SINTÁCTICO"),
    ("Bloque sin propiedades", "effect Eco { }", "SINTÁCTICO"),
    ("Falta identificador después de ->", "mix Salida = Melodia -> ;", "SINTÁCTICO"),
    ("BPM con token FLOAT en lugar de INT", "track Melodia { bpm: 120.5; }", "SINTÁCTICO"),
    ("Separador de nota duplicado", "track Melodia { notes: [C4::1/1]; }", "SINTÁCTICO"),
    ("Texto extra al final del programa", "track Melodia { bpm: 120; } sobrante", "SINTÁCTICO"),
    ("Carácter no reconocido", "track Melodia { bpm: 120; } @", "LÉXICO"),
    ("Identificador con tilde no admitida por el lexer", "track PadAtmósfera { bpm: 120; }", "LÉXICO"),
]

for titulo, codigo, fase in casos_sintacticos_invalidos:
    resultado = probar_analizador_sintactico(codigo, titulo)
    assert not resultado["valido"], titulo
    assert resultado["arbol"] is None, titulo
    assert resultado["errores"], titulo
    assert all(e["fase"] == fase for e in resultado["errores"]), titulo

print(f"Pruebas de rechazo superadas: {len(casos_sintacticos_invalidos)}.")

Programa vacío
Código de entrada:

Sin errores sintácticos.
Árbol sintáctico: (program <EOF>)

..................................................



AssertionError: Programa vacío

Pruebas de separación entre sintaxis y semántica

Estas entradas tienen estructura válida, aunque el analizador semántico existente debe reportar una referencia no declarada (SEM-02), una nota fuera del rango audible (SEM-03) o un compás incompleto (SEM-04). No se rechazan en la fase sintáctica.

In [ ]:
casos_sintacticos_semantica = [
    ("SEM-02: referencia no declarada", "mix Salida = Desconocida;"),
    ("SEM-03: nota fuera del rango audible", "track Grave { notes: [C0:1/1]; }"),
    ("SEM-04: duraciones que no completan el compás", "track Incompleta { notes: [C4:1/4, E4:1/2]; }"),
]

for titulo, codigo in casos_sintacticos_semantica:
    resultado = probar_analizador_sintactico(
        codigo, titulo, mostrar_arbol=False
    )
    assert resultado["valido"], titulo
    print("La comprobación indicada corresponde al analizador semántico.\n")

total_pruebas = (
    len(casos_sintacticos_validos) + len(casos_sintacticos_invalidos)
    + len(casos_sintacticos_semantica)
)
print(f"Total: {total_pruebas} pruebas del analizador sintáctico superadas.")

**ANALIZADOR SEMÁNTICO**

Errores semánticos verificados:

| Código | Error | Regla |
|---|---|---|
| SEM-01 | Identificador redeclarado | Cada `track`, `effect` y `mix` tiene un nombre único |
| SEM-02 | Referencia no declarada | La pista y los efectos de un `mix` deben estar declarados |
| SEM-03 | Nota fuera del rango audible | 20 Hz ≤ frecuencia ≤ 20 000 Hz |
| SEM-04 | Compás incompleto | Las duraciones de `notes` deben completar compases enteros |
| SEM-05 | Categoría incorrecta en `mix` | La fuente debe ser un `track` y la cadena solo admite `effect` |
| SEM-06 | Duración inválida | El numerador y el denominador de la duración deben ser mayores que 0 |

Usa el mismo `SoundSpecParser.g4` compilado en la sección sintáctica.

Analizador semántico (tabla de símbolos y dos recorridos del árbol: el primero registra las declaraciones y el segundo valida notas y mezclas)

In [ ]:
from fractions import Fraction
from antlr4 import ParseTreeWalker
from SoundSpecParserListener import SoundSpecParserListener

SEMITONO = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}
ALTERACION = {'': 0, '#': 1, 'b': -1}

def frecuencia(nota): # C#4
    # f = 440 * 2^((midi - 69) / 12), con A4 = 440 Hz (midi 69)
    midi = 12 * (int(nota[-1]) + 1) + SEMITONO[nota[0]] + ALTERACION[nota[1:-1]]
    return 440 * 2 ** ((midi - 69) / 12)

class Declaraciones(SoundSpecParserListener):
    # Pasada 1: llena la tabla de símbolos {nombre: (categoría, línea)} y detecta SEM-01
    def __init__(self, tabla, errores):
        self.tabla, self.errores = tabla, errores

    def declarar(self, token, categoria):
        if token.text in self.tabla:
            cat, linea = self.tabla[token.text]
            self.errores.append(f"[SEM-01] línea {token.line}: '{token.text}' ya fue declarado como {cat} en la línea {linea}.")
        else:
            self.tabla[token.text] = (categoria, token.line)

    def enterTrackDecl(self, ctx):  self.declarar(ctx.ID().symbol, 'track')
    def enterEffectDecl(self, ctx): self.declarar(ctx.ID().symbol, 'effect')
    def enterMixRule(self, ctx):    self.declarar(ctx.ID(0).symbol, 'mix')

class Verificador(SoundSpecParserListener):
    # Pasada 2: detecta SEM-02 a SEM-06
    def __init__(self, tabla, errores):
        self.tabla, self.errores = tabla, errores

    def enterNoteList(self, ctx):
        total = Fraction(0)
        for item in ctx.noteItem():
            nota = item.NOTE().getText()
            linea = item.start.line
            f = frecuencia(nota)
            if not 20 <= f <= 20000:
                self.errores.append(f"[SEM-03] línea {linea}: la nota {nota} ({f:.2f} Hz) está fuera del rango audible.")
            duracion = item.FRACTION().getText()
            num, den = map(int, duracion.split('/'))
            if num == 0 or den == 0:
                # La nota no se suma al compás porque su duración no es válida
                self.errores.append(f"[SEM-06] línea {linea}: la nota {nota} tiene una duración inválida ({duracion}).")
            else:
                total += Fraction(num, den)
        if total.denominator != 1:
            self.errores.append(f"[SEM-04] línea {ctx.start.line}: las duraciones suman {total} de compás; falta {1 - total % 1}.")

    def enterMixRule(self, ctx):
        ids, linea = ctx.ID(), ctx.start.line
        fuente = ids[1].getText()
        if fuente not in self.tabla:
            self.errores.append(f"[SEM-02] línea {linea}: la pista '{fuente}' no ha sido declarada.")
        elif self.tabla[fuente][0] != 'track':
            self.errores.append(f"[SEM-05] línea {linea}: '{fuente}' fue declarado como {self.tabla[fuente][0]}, pero la fuente de un mix debe ser un track.")
        for efecto in ids[2:]:
            nombre = efecto.getText()
            if nombre not in self.tabla:
                self.errores.append(f"[SEM-02] línea {linea}: el efecto '{nombre}' no ha sido declarado.")
            elif self.tabla[nombre][0] != 'effect':
                self.errores.append(f"[SEM-05] línea {linea}: '{nombre}' fue declarado como {self.tabla[nombre][0]}, pero la cadena de un mix solo admite effects.")

def analizador_semantico(cod_fuente, Nprueba):
    print(Nprueba)
    print("\nCódigo de entrada:" + cod_fuente)
    resultado = analizar_sintaxis(cod_fuente)
    if not resultado["valido"]:
        errores = [f"[{e['fase']}] línea {e['linea']}, columna {e['columna']}: {e['mensaje']}"
                   for e in resultado["errores"]]
        errores.append("No se ejecuta el análisis semántico.")
    else:
        tabla, errores = {}, []
        ParseTreeWalker().walk(Declaraciones(tabla, errores), resultado["arbol"])
        ParseTreeWalker().walk(Verificador(tabla, errores), resultado["arbol"])
        errores.sort(key=lambda e: int(e.split("línea ")[1].split(":")[0]))

    print("\n".join(errores) if errores else "Sin errores semánticos.")
    print("\n" + "." * 50 + "\n")

Pruebas del analizador semántico

In [ ]:
prueba1 = """
mix Song = BassLine -> Reverb1;
track BassLine {
    bpm: 120;
    wave: SAWTOOTH;
    notes: [ C#4 1/2, Db4 1/4, Db4 1/4 ];
}
effect Reverb1 { delay: 200ms; decay: 0.5; }
"""

prueba2 = """
track Melodia { notes: [ C4 1/1 ]; }
effect Melodia { decay: 0.5; }
"""

prueba3 = """
track Melodia { notes: [ C4 1/1 ]; }
effect Eco { delay: 300ms; }
mix Salida = Guitarra -> Chorus;
mix Final = Eco;
mix Doble = Melodia -> Melodia;
"""

prueba4 = """
track Alteraciones { notes: [ C#4 1/2, Db4 1/4, D#0 1/4 ]; }
"""

prueba5 = """
track Melodia { notes: [ C4 1/4, D4 1/4, E4 1/4 ]; }
track Ritmo { notes: [ C4 1/0, D4 0/4, E4 1/1 ]; }
"""

prueba6 = """
track Melodia { wave = SINE; }
"""

prueba7 = """
track Melodia { bpm: 120; } @
"""

analizador_semantico(prueba1, "PRUEBA 1: Programa correcto (mix antes de sus declaraciones, con sostenidos y bemoles)")
analizador_semantico(prueba2, "PRUEBA 2: Identificador redeclarado (SEM-01)")
analizador_semantico(prueba3, "PRUEBA 3: Referencias no declaradas (SEM-02) y de categoría incorrecta (SEM-05)")
analizador_semantico(prueba4, "PRUEBA 4: Nota fuera del rango audible con alteraciones (SEM-03)")
analizador_semantico(prueba5, "PRUEBA 5: Compás incompleto (SEM-04) y duraciones inválidas (SEM-06)")
analizador_semantico(prueba6, "PRUEBA 6: Error sintáctico")
analizador_semantico(prueba7, "PRUEBA 7: Error léxico (carácter no reconocido)")